In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

splits = pd.read_csv(r"D:\AgriVision\data\splits.csv")
train_df = splits[splits.split == "train"]
val_df = splits[splits.split == "val"]
test_df = splits[splits.split == "test"]

classes = sorted(splits["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

IMG_SIZE, BATCH = 224, 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, label   # no /255 here: MobileNetV2 preprocessing is inside the model

def make_ds(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["label"].map(class_to_idx).values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)

y_train = train_df["label"].map(class_to_idx).values
w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(w))
print(classes)

['BrownRust', 'Healthy', 'Mildew', 'Septoria', 'YellowRust']


In [2]:
from tensorflow import keras
from tensorflow.keras import layers

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

base = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(classes), activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10,
                    class_weight=class_weights)

Epoch 1/10


d:\AgriVision\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


In [ ]:
loss, acc = model.evaluate(test_ds)
print("Test accuracy:", acc)
model.save(r"D:\AgriVision\models\leaf_model.keras")